
# 00 — End-to-End Reproducibility Validation

## Urban Infrastructure Pulse Intelligence

### Objective
Verify that the complete 2023–2025 NYC 311 data pipeline is reproducible, with no accidental reuse of artifacts from the historical 100K-sample analysis.

### Validation Principles
- Preserve raw data and historical backups.
- Verify dataset scope and source provenance.
- Rebuild outputs in dependency order.
- Check geographic identifiers and feature integrity.
- Record output file hashes.
- Do not silently reuse stale PCA or K-Means models.
- Retain all geographic areas in primary clustering.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import json

PROJECT_ROOT = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "config" / "config.py").is_file()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Open the project in VS Code."
    )

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MASTER_FILE = (
    PROCESSED_DIR / "nyc311_infrastructure_cleaned.csv"
)

if not MASTER_FILE.is_file():
    raise FileNotFoundError(MASTER_FILE)

print("Project root:", PROJECT_ROOT)
print("Master dataset:", MASTER_FILE)
print("File size (MB):", round(
    MASTER_FILE.stat().st_size / 1024**2, 2
))

Project root: d:\Urban-Infrastructure-Pulse-Intelligence
Master dataset: d:\Urban-Infrastructure-Pulse-Intelligence\data\processed\nyc311_infrastructure_cleaned.csv
File size (MB): 447.82


In [2]:
df_audit = pd.read_csv(
    MASTER_FILE,
    low_memory=False
)

required_columns = [
    "unique_key",
    "created_date",
    "complaint_type",
    "borough",
    "incident_zip"
]

missing = [
    col for col in required_columns
    if col not in df_audit.columns
]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

created_dates = pd.to_datetime(
    df_audit["created_date"],
    errors="coerce"
)

start = pd.Timestamp("2023-01-01")
end = pd.Timestamp("2026-01-01")

audit = {
    "records": len(df_audit),
    "columns": df_audit.shape[1],
    "complaint_types": df_audit["complaint_type"].nunique(),
    "duplicate_unique_keys": int(
        df_audit["unique_key"].duplicated().sum()
    ),
    "invalid_dates": int(created_dates.isna().sum()),
    "outside_study_dates": int(
        ((created_dates < start) | (created_dates >= end)).sum()
    ),
    "earliest_date": str(created_dates.min()),
    "latest_date": str(created_dates.max())
}

print("FULL DATASET AUDIT")
print("=" * 50)

for key, value in audit.items():
    print(f"{key}: {value}")

# Historical full-data run reference checks
assert audit["records"] == 685284, "Record count differs"
assert audit["columns"] == 53, "Column count differs"
assert audit["complaint_types"] == 15, "Category count differs"
assert audit["duplicate_unique_keys"] == 0
assert audit["invalid_dates"] == 0
assert audit["outside_study_dates"] == 0

print("\nFULL-DATA SIGNATURE: PASSED")

FULL DATASET AUDIT
records: 685284
columns: 53
complaint_types: 15
duplicate_unique_keys: 0
invalid_dates: 0
outside_study_dates: 0
earliest_date: 2023-01-01 00:00:57
latest_date: 2025-12-31 23:57:00

FULL-DATA SIGNATURE: PASSED


In [3]:
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()

audit["master_file_sha256"] = sha256_file(MASTER_FILE)

AUDIT_DIR = (
    PROJECT_ROOT / "outputs" / "reports" / "reproducibility"
)

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

audit_path = AUDIT_DIR / "master_dataset_audit.json"

with open(audit_path, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=4)

print("Master SHA256:", audit["master_file_sha256"])
print("Audit saved:", audit_path)

Master SHA256: 8635dd08754013e26d154208225de0eca386564df6a2df127bd500eacd4b0d08
Audit saved: d:\Urban-Infrastructure-Pulse-Intelligence\outputs\reports\reproducibility\master_dataset_audit.json
